# Audit the ERalpha input before native modeling

This notebook executes the first validation gate for the existing regression
snapshot. It records **blocked chemical recognition**, without assigning a fit
score or claiming a completed native pharmacophore. The historical feature-
presence regression still uses the legacy modeler.

Run from a source checkout with compatible development providers. The fixture and
case-specific script live in `tests/data/` and `devtools/`; they are not promised
as installed package data. Start the kernel within the checkout, or set
`PHMT_REPOSITORY_ROOT` to the checkout's absolute path before starting it.

The recorded run uses Python 3.13.14, MolSysMT source revision
`4d490427e38c5836be82472348fdf61934442bda`, PyUnitWizard
`2ffe1885675f47c76af03c08e51bc889a5e99a05` and existing scientific dependencies.
This is source-tree evidence, not stable-distribution acceptance.

Owning work: [PharmacophoreMT #22](https://github.com/uibcdf/pharmacophoremt/issues/22)
and [MolSysMT #298](https://github.com/uibcdf/molsysmt/issues/298).

In [1]:
import json
import os
import sys
from pathlib import Path

start = Path(os.environ.get('PHMT_REPOSITORY_ROOT', Path.cwd())).resolve()
root = next((p for p in (start, *start.parents)
             if (p / 'tests/data/eralpha_audit.json').is_file()), None)
if root is None:
    raise RuntimeError('Set PHMT_REPOSITORY_ROOT to the PharmacophoreMT checkout.')
sys.path.insert(0, str(root))
from devtools.audit_eralpha import audit

manifest = json.loads((root / 'tests/data/eralpha_audit.json').read_text())
print(json.dumps({key: manifest[key] for key in
                 ['fixture', 'sha256', 'description', 'provenance_status']}, indent=2))

{
  "fixture": "tests/data/eralpha_complex.pdb",
  "sha256": "90e3159bc03424d976742c96c8cc2d2b32c9e1abf140d4652b70518b06a775f1",
  "description": "Prepared OpenMM snapshot historically labeled ERalpha/estradiol",
  "provenance_status": "derivation_and_chemical_identity_unverified"
}


## Execute the provider gate

All molecular conversion, selection, extraction, coordinates and chemical-state
access go through public MolSysMT APIs. Counts describe the provider's loaded
graph, which can include its default connectivity reconstruction. The audit
aggregates observations for this particular snapshot; general molecular readiness
belongs to [MolSysMT #217](https://github.com/uibcdf/molsysmt/issues/217) and receptor
coverage to [#218](https://github.com/uibcdf/molsysmt/issues/218).

The file has an OpenMM preparation header, explicit hydrogens and solvent. Its
unnamed ligand cannot be located by automatic small-molecule classification.
The fixture-specific selection is therefore qualified by the file checksum and
retained source atom identities.

In [2]:
report = audit()
summary = {
    'python': sys.version.split()[0],
    'molsysmt_version': report['molsysmt_version'],
    'counts': report['counts'],
    'ligand_element_counts': report['ligand_element_counts'],
    'automatic_small_molecule_selection_count': report['automatic_small_molecule_selection_count'],
    'coordinates_finite': report['coordinates_finite'],
    'source_unchanged': report['source_unchanged'],
    'parts': {
        part: {
            route: {key: observation[key] for key in
                    ['n_atoms', 'n_bonds', 'connectivity_completeness',
                     'recognition_status', 'diagnostic']}
            for route, observation in routes.items()
        }
        for part, routes in report['parts'].items()
    },
}
print(json.dumps(summary, indent=2))

{
  "python": "3.13.14",
  "molsysmt_version": "0.22.4+118.g03b318549.dirty",
  "counts": {
    "n_atoms": 54437,
    "n_groups": 17032,
    "n_molecules": 16783,
    "n_structures": 1
  },
  "ligand_element_counts": {
    "C": 18,
    "O": 2,
    "H": 24
  },
  "automatic_small_molecule_selection_count": 0,
  "coordinates_finite": true,
  "source_unchanged": true,
  "parts": {
    "ligand": {
      "native_pdb": {
        "n_atoms": 44,
        "n_bonds": 47,
        "connectivity_completeness": "partial",
        "recognition_status": "blocked",
        "diagnostic": {
          "code": "MSM-ERR-STRUCT-003",
          "message": "Structural inconsistency detected: Chemical graph analysis requires connectivity declared complete.. Ensure that the atoms, residues, or frames match between the systems being compared or merged. Docs: https://www.uibcdf.org/MolSysMT"
        }
      },
      "pdb_text_rdkit_roundtrip": {
        "n_atoms": 44,
        "n_bonds": 47,
        "connectivity_co

In [3]:
assert report['sha256'] == manifest['sha256']
assert report['counts'] == manifest['expected_counts']
assert report['ligand_atom_names'] == manifest['ligand']['atom_names']
assert report['ligand_element_counts'] == manifest['ligand']['element_counts']
assert report['source_unchanged'] and report['coordinates_finite']
for part, routes in report['parts'].items():
    for route, observation in routes.items():
        assert observation['n_atoms'] == manifest[part]['n_atoms']
        assert observation['n_bonds'] == manifest[part]['n_bonds']
        assert observation['recognition_status'] == 'blocked'
        assert observation['diagnostic']['code'] == 'MSM-ERR-STRUCT-003'
        assert 'fit_value' not in observation
        if route == 'pdb_text_rdkit_roundtrip':
            assert set(observation['bond_orders']) == {0}
print('Input audit passed. Native chemical preparation remains unresolved.')

Input audit passed. Native chemical preparation remains unresolved.


## What this result establishes

Native PDB import records partial connectivity and absent declared bond orders and
formal charges. Public PDB-text → RDKit → native conversion preserves atom/bond
counts but retains unsupported order-zero bonds. MolSysMT rejects both inputs.
A `complete` connectivity flag alone does not justify chemical feature recognition.
Neither failure is a geometrically nonmatching ligand or a biological inactive.

The [historical dataset label, PDB 1QKU](https://www.rcsb.org/structure/1QKU), refers
to the wild-type human estrogen receptor ligand-binding domain with estradiol.
The local prepared snapshot's derivation from that entry is unverified. C18H24O2
is compatible with estradiol, but does not verify connectivity or stereochemistry.
[RCSB CCD EST](https://www.rcsb.org/ligand/EST) is a candidate authoritative template;
it has not been applied. The associated
[Gangloff et al. publication (2001)](https://doi.org/10.1074/jbc.M009870200) is
reference context, not validation of this snapshot.

These reference-data sources must remain separate from executed software/method
attribution. See the [Ackredit recipe](attribution.md) for detached calculation
bibliographies once a native calculation completes.

## Controls after provider-supported preparation

First recover the preparation history or adopt an independently sourced input.
Choose the ligand chemical state and review an explicit template atom map,
including hydrogens, conflicts, orders, aromaticity, charges and coordinates.
MolSysMT owns this operation; PharmacophoreMT consumes the resulting system.

Then execute the following controls, retaining units, parameters and source
identities in the query/results:

| Control | Expected evidence |
| --- | --- |
| Reference placed against its own native query | Matched, unit geometric coverage |
| Same ligand displaced by 2 nm, without alignment | Essential sites missing, placed-pose rejection |
| Shared rigid transform of query and input | Same geometric outcome |
| Dedicated donor direction and aromatic-normal changes | Rejection according to the declared angular criteria |
| JSON roundtrip | Same features, shapes, units, evidence and evaluation |
| Explicit Ackredit session | Detached references for executed software/methods |
| Provider-observed receptor interactions | Named detector profile, evaluated coverage and retained observations |

A proposed initial site radius of 0.10 nm and direction tolerance of 20 degrees
are geometric test choices; compare 0.15/0.20 nm sensitivity before interpreting
results. They are not established binding cutoffs. Review expected feature counts
after chemical preparation; do not copy legacy counts as a native scientific oracle.
Chemical features require hypothesis curation before being treated as binding roles.

Rigid search can recover a displaced copy, so that displacement is a negative
control for **placed evaluation** only. Geometric self-consistency is not evidence
of affinity, enrichment or performance. No ERalpha positive native validation is
claimed by the audit above.

## Independently acquired 1QKU / CCD EST input

The source bytes in `tests/data/eralpha_rcsb/` were acquired from official RCSB
services on 2026-10-03 and qualified with URLs, checksums, file sizes and entry
revision. This gives the new case a documented origin; it does not recover the
old OpenMM snapshot's history. See [RCSB 1QKU](https://www.rcsb.org/structure/1QKU),
[CCD EST](https://www.rcsb.org/ligand/EST) and the
[official download services](https://www.rcsb.org/docs/programmatic-access/file-download-services).

Select the ligand by **label asymmetry ID D**, represented by MolSysMT `chain_id`.
Its **author chain is A**, residue 600. The 20 observed heavy-atom names agree with
the CCD definition, which also contains 24 H. Name agreement is candidate
correspondence evidence; an exhaustive chemical map has not been validated.

In [4]:
from devtools.audit_eralpha_sources import audit as audit_sources

source_manifest = json.loads((root / 'tests/data/eralpha_rcsb/manifest.json').read_text())
source_report = audit_sources()
assert source_report['entry_id'] == '1QKU'
assert source_report['counts'] == source_manifest['expected_counts']
assert source_report['checksums'] == {
    name: data['sha256'] for name, data in source_manifest['files'].items()
}
assert source_report['author_asym_ids'] == ['A']
assert source_report['author_residue_ids'] == ['600']
assert source_report['ligand_element_counts'] == {'C': 18, 'O': 2}
assert source_report['ccd_element_counts'] == {'C': 18, 'H': 24, 'O': 2}
assert source_report['heavy_atom_names_agree']
assert source_report['source_unchanged'] and source_report['files_unchanged']
assert source_report['recognition_status'] == 'blocked'
assert source_report['chemical_mapping_status'] == 'not_validated'
assert 'fit_value' not in source_report
print(json.dumps(source_report, indent=2))

{
  "schema": "pharmacophoremt.eralpha_source_audit@1",
  "molsysmt_version": "0.22.4+118.g03b318549.dirty",
  "checksums": {
    "1qku.cif": "8f06e49437b661728c5ca1e3427a4f1bdce9aee1aa9bbe27f1eab4fb2d17f02e",
    "EST.cif": "8b80ee77d734e62a82a27571bc1de261e8ea4cf9d053490d10a39c614a69d3a8",
    "1qku_EST_D.sdf": "cced6aff1f507dc0520e3be8bb6b5a67592099461d3f77067ed2e362ffbf893f"
  },
  "entry_id": "1QKU",
  "observed_revision": {
    "ordinal": "5",
    "data_content_type": "Structure model",
    "major_revision": "1",
    "minor_revision": "4",
    "revision_date": "2024-05-08"
  },
  "counts": {
    "n_atoms": 6596,
    "n_groups": 1343,
    "n_structures": 1
  },
  "source_atom_indices": [
    5940,
    5941,
    5942,
    5943,
    5944,
    5945,
    5946,
    5947,
    5948,
    5949,
    5950,
    5951,
    5952,
    5953,
    5954,
    5955,
    5956,
    5957,
    5958,
    5959
  ],
  "ligand_atom_names": [
    "C1",
    "C2",
    "C3",
    "O3",
    "C4",
    "C5",
    "C6",

## Minimum preparation contract

The proposed tools in [MolSysMT #298](https://github.com/uibcdf/molsysmt/issues/298)
separate a read-only chemical-template assessment from application to a new
native system. The first implementation requires an explicit exhaustive atom map,
compatible declared chemistry, resolved states and a complete supported template.
It must report conflicts, preserve source coordinates/identity and retain the
template's origin. These tools are proposed; this notebook does not call a
placeholder API or claim that template application is already implemented.

The observed ligand has no H coordinates. Applying chemical assignments does not
supply the missing donor-H geometry; hydrogen generation is a distinct provider
operation. CCD model/ideal H positions have not been transplanted to the complex.
The retained ModelServer SDF is an auxiliary source export with an unversioned
CTAB counts line, rejected by the native SDF probe at clean MolSysMT source
`e9f135c2962aeefdb08135a7bc567d2d08bf27c5`. Its bytes are not repaired locally.

The source-curation gate is now reproducible. Chemical preparation, positive
native evaluation and the geometric controls above remain the next acceptance
steps. Neither input audit is an ERalpha biological benchmark.